# 컨텍스트 분리와 Isolate

같은 상태에 섞인 여러 업무 메시지에서 현재 업무에 필요한 내용만 모델 요청으로 분리하려면 어떻게 할까요?

## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/08-컨텍스트-분리와-Isolate.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.

## 패키지 버전 확인

In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 메시지 수집 모델 준비

In [2]:
# 이 셀은 메시지 수집 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration,ChatResult
class CapturingModel(BaseChatModel):
    seen:list[str]=[]
    @property
    def _llm_type(self) -> str: return "capture"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        self.seen=[m.text for m in messages]
        return ChatResult(generations=[ChatGeneration(message=AIMessage("확인 완료"))])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "CapturingModel": return self

## 업무가 섞인 메시지 준비

In [3]:
# 이 셀은 업무가 섞인 메시지 준비 작업을 수행하고 확인할 값을 출력합니다.
from langchain_core.messages import HumanMessage,AIMessage
mixed=[
 HumanMessage("대출은 5권",additional_kwargs={"topic":"loan"}),AIMessage("기간은 14일",additional_kwargs={"topic":"loan"}),
 HumanMessage("급여 자료는 내부 문서",additional_kwargs={"topic":"hr"}),AIMessage("인사팀 확인",additional_kwargs={"topic":"hr"}),
 HumanMessage("대출 연장은?",additional_kwargs={"topic":"loan"})]
print("state_messages:",len(mixed))

state_messages: 5


## Before: 전체 메시지 전달

In [4]:
# 이 셀은 Before: 전체 메시지 전달 작업을 수행하고 확인할 값을 출력합니다.
before_model=CapturingModel()
create_agent(before_model,tools=[]).invoke({"messages":mixed})
print("model_messages:",before_model.seen)
assert len(before_model.seen)==5 and any("급여" in text for text in before_model.seen)

model_messages: ['대출은 5권', '기간은 14일', '급여 자료는 내부 문서', '인사팀 확인', '대출 연장은?']


## After: 업무별 요청 분리

In [5]:
# 이 셀은 After: 업무별 요청 분리 작업을 수행하고 확인할 값을 출력합니다.
class IsolateMiddleware(AgentMiddleware):
    def __init__(self, topic: str) -> None: self.topic=topic
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        selected=[m for m in request.messages if m.additional_kwargs.get("topic")==self.topic]
        return handler(request.override(messages=selected))
after_model=CapturingModel()
create_agent(after_model,tools=[],middleware=[IsolateMiddleware("loan")]).invoke({"messages":mixed})
print("model_messages:",after_model.seen)
assert len(after_model.seen)==3 and all("급여" not in text for text in after_model.seen)

model_messages: ['대출은 5권', '기간은 14일', '대출 연장은?']


## 상태와 요청 범위 비교

In [6]:
# 이 셀은 상태와 요청 범위 비교 작업을 수행하고 확인할 값을 출력합니다.
print({"stored_messages":len(mixed),"selected_messages":len(after_model.seen),"security_boundary":False,"estimated_cost_usd":0.0})

{'stored_messages': 5, 'selected_messages': 3, 'security_boundary': False, 'estimated_cost_usd': 0.0}
